# 02. Echoes original_audio ↔ FMA REAL 매칭 확인

## 목적

앞 단계에서 Echoes TTA 데이터를 정제하여 다음을 확인했다.

- Clean TTA: **3,162개**
- `original_audio` 그룹: **296개**
- 실제 TTA 파일 누락: **0개**

이번 노트북에서는 이 **296개의 `original_audio` 이름을 FMA `tracks.csv`와 연결**한다.

최종 목표:

```text
Echoes original_audio
        ↓
FMA track_id
        ↓
REAL 인간 음악
```

이 단계에서는 아직 실제 FMA 오디오를 다운로드하지 않는다. 먼저 metadata만 이용해 어떤 FMA track_id가 필요한지 확인한다.


## 0. 프로젝트 경로 설정

노트북을 `project/` 또는 `project/notebooks/`에서 실행해도 프로젝트 루트를 자동으로 찾도록 한다.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    if (PROJECT_ROOT.parent / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

ECHOES_MANIFEST = PROJECT_ROOT / "data/raw/Echoes/Echoes/dataset_manifest.csv"
FMA_TRACKS = PROJECT_ROOT / "data/raw/FMA/fma_metadata/tracks.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Echoes manifest exists:", ECHOES_MANIFEST.exists())
print("FMA tracks.csv exists:", FMA_TRACKS.exists())


## 1. Echoes Clean TTA 다시 생성

앞 노트북과 같은 정제 규칙을 적용한다.

1. `type == "TTA"`만 사용
2. 동일 `path_in_dataset`을 여러 행이 공유하는 경우 모두 제외
3. 남은 `original_audio` 고유값을 추출


In [ ]:
echoes = pd.read_csv(ECHOES_MANIFEST)

tta = echoes[echoes["type"] == "TTA"].copy()

dup_mask = tta["path_in_dataset"].duplicated(keep=False)
tta_clean = tta[~dup_mask].copy()

originals = (
    tta_clean[["original_audio", "genre"]]
    .drop_duplicates("original_audio")
    .sort_values("original_audio")
    .reset_index(drop=True)
)

print("Original TTA rows :", len(tta))
print("Excluded rows     :", int(dup_mask.sum()))
print("Clean TTA rows    :", len(tta_clean))
print("Original groups   :", len(originals))

display(originals.head(10))


### 기대 결과

```text
Original TTA rows : 3165
Excluded rows     : 3
Clean TTA rows    : 3162
Original groups   : 296
```


## 2. FMA tracks.csv 로드

FMA `tracks.csv`는 MultiIndex 컬럼을 사용한다. 매칭에 필요한 정보만 별도 DataFrame으로 만든다.

- `track_id`
- title
- artist
- genre_top
- license
- duration
- subset


In [ ]:
fma = pd.read_csv(
    FMA_TRACKS,
    header=[0, 1],
    index_col=0
)

fma_simple = pd.DataFrame({
    "track_id": fma.index.astype(int),
    "title": fma[("track", "title")].values,
    "artist": fma[("artist", "name")].values,
    "genre_top": fma[("track", "genre_top")].values,
    "license": fma[("track", "license")].values,
    "duration": fma[("track", "duration")].values,
    "subset": fma[("set", "subset")].values,
})

print("FMA tracks:", len(fma_simple))
display(fma_simple.head())


## 3. 문자열 정규화

Echoes의 `original_audio`는 보통 `곡 제목 - 아티스트` 형식이다.

FMA에서도 `track.title + " - " + artist.name`을 만들어 비교한다.

대소문자, Unicode 표현, 앞뒤 공백, 중복 공백만 정규화하고, 우선은 보수적인 exact matching을 수행한다.


In [ ]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = unicodedata.normalize("NFKC", str(x))
    x = x.strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

fma_simple["fma_name"] = (
    fma_simple["title"].fillna("").astype(str).str.strip()
    + " - "
    + fma_simple["artist"].fillna("").astype(str).str.strip()
)

fma_simple["match_key"] = fma_simple["fma_name"].map(normalize_text)
originals["match_key"] = originals["original_audio"].map(normalize_text)

display(fma_simple[["track_id","title","artist","fma_name","match_key"]].head())


## 4. Exact matching 후보 개수 확인

각 Echoes `original_audio`에 대해 FMA에서 동일한 `match_key`가 몇 개 존재하는지 센다.

- **1개 후보**: 거의 바로 연결 가능
- **2개 이상 후보**: 추가 규칙 필요
- **0개 후보**: 문자열 차이 또는 metadata 불일치 조사 필요


In [ ]:
candidate_counts = (
    fma_simple.groupby("match_key")
    .size()
    .rename("candidate_count")
)

match_summary = originals.merge(
    candidate_counts,
    left_on="match_key",
    right_index=True,
    how="left"
)

match_summary["candidate_count"] = (
    match_summary["candidate_count"]
    .fillna(0)
    .astype(int)
)

print("===== EXACT MATCH SUMMARY =====")
print(match_summary["candidate_count"].value_counts().sort_index())

print("\nExactly 1 candidate :", int((match_summary["candidate_count"] == 1).sum()))
print("Multiple candidates :", int((match_summary["candidate_count"] > 1).sum()))
print("No candidate        :", int((match_summary["candidate_count"] == 0).sum()))
print("Total               :", len(match_summary))


## 5. 매칭되지 않은 원곡 확인

In [ ]:
no_match = match_summary[
    match_summary["candidate_count"] == 0
].copy()

print("No-match count:", len(no_match))
display(no_match[["original_audio", "genre"]])


## 6. 복수 후보 원곡 확인

같은 `곡 제목 - 아티스트`가 FMA에 여러 번 존재할 수 있다. 이 경우 자동으로 첫 번째 곡을 선택하면 안 된다.

후보들의 `track_id`, `genre_top`, `license`, `duration`, `subset`을 함께 확인한다.


In [ ]:
multiple = match_summary[
    match_summary["candidate_count"] > 1
].copy()

print("Multiple-match original_audio count:", len(multiple))
display(multiple[["original_audio", "genre", "candidate_count"]])


In [ ]:
multi_candidates = multiple[
    ["original_audio", "genre", "match_key"]
].merge(
    fma_simple[
        [
            "track_id","title","artist","genre_top","license",
            "duration","subset","fma_name","match_key"
        ]
    ],
    on="match_key",
    how="left"
).sort_values(["original_audio", "track_id"])

display(multi_candidates)


## 7. 정확히 1개 후보인 원곡의 매칭 결과 확인

In [ ]:
single = match_summary[
    match_summary["candidate_count"] == 1
][["original_audio", "genre", "match_key"]].copy()

single_matches = single.merge(
    fma_simple[
        [
            "track_id","title","artist","genre_top","license",
            "duration","subset","fma_name","match_key"
        ]
    ],
    on="match_key",
    how="left"
)

print("Single exact matches:", len(single_matches))
display(single_matches.head(20))


## 8. 장르 일치 여부 간단 점검

Echoes genre와 FMA `genre_top`의 일치 여부를 확인한다.

장르가 다르다고 바로 제거하지는 않는다. 두 데이터셋의 장르 분류 기준이 완전히 같다고 보장할 수 없으므로 품질 점검용으로만 사용한다.


In [ ]:
single_matches["genre_match"] = (
    single_matches["genre"].astype(str).str.lower()
    == single_matches["genre_top"].astype(str).str.lower()
)

print(single_matches["genre_match"].value_counts(dropna=False))

display(
    single_matches.loc[
        ~single_matches["genre_match"],
        ["original_audio","genre","track_id","genre_top","title","artist"]
    ].head(30)
)


## 9. 실행 후 기록할 핵심 숫자

```text
Exactly 1 candidate : ?
Multiple candidates : ?
No candidate        : ?
```

세 숫자의 합은 **296**이어야 한다.

이 결과를 확인한 뒤:
1. 복수 후보 선택 규칙 확정
2. 무매칭 원곡 보정
3. 최종 `original_audio → FMA track_id` 매핑 확정

으로 진행한다.

---

## 10. 이후 개발 파일

매칭 규칙이 확정되면 실제 개발 스크립트를 별도로 만든다.

예정:
```text
src/01_build_master_manifest.py
```

지금은 아직 후보 검증 단계이므로 자동 선택 로직을 먼저 확정하지 않는다.
